Step 1: Importing Packages

In [7]:
import numpy as np
import pandas as pd
import yfinance as yf

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

Step 2: Downloading data

In [9]:
raw = yf.download(
    "RELIANCE.NS",
    period="10y",
    interval="1d",
    auto_adjust=True,
    progress=False,
    multi_level_index=False,
)

if raw.empty:
    raise ValueError("No Reliance data downloaded. Check your connection or ticker.")

if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)

data = raw[["Open", "High", "Low", "Close", "Volume"]].copy()
data = data.sort_index()
data.index = pd.DatetimeIndex(data.index).tz_localize(None).normalize()

close = data["Close"]

In [10]:
#plotting the returns
import matplotlib.pyplot as plt

plot_start = model_data.index.min()
plot_end = model_data.index.max()
prices = data.loc[plot_start:plot_end, "Close"]

plt.style.use("dark_background")
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(
    prices.index,
    prices,
    color="#58C4DD",
    linewidth=1.8,
    label="Adjusted closing price",
)

ax.axvline(
    pd.Timestamp("2024-10-29"),
    color="#FF6B6B",
    linestyle="--",
    linewidth=1.5,
    label="Final test period starts",
)

ax.set_title(
    "Reliance Industries Adjusted Closing Price",
    fontsize=17,
    fontweight="bold",
    pad=16,
)
ax.set_xlabel("Date", fontsize=11)
ax.set_ylabel("Adjusted closing price (₹)", fontsize=11)
ax.grid(True, color="white", alpha=0.12)
ax.legend(frameon=False)

fig.patch.set_facecolor("#111827")
ax.set_facecolor("#111827")
plt.tight_layout()
plt.show()

NameError: name 'model_data' is not defined

Step 3: Calculate the future return and choose the flat band

In [11]:
future_return_pct = ((close.shift(-5) / close) - 1) * 100
future_return_pct = future_return_pct.rename("future_return_pct")
known_returns = future_return_pct.dropna()

test_start = pd.Timestamp("2024-10-29")

# Remove pre-test rows whose five-session target windows could overlap the test period.
pretest_end = max(0, known_returns.index.searchsorted(test_start) - 5)
pretest_returns = known_returns.iloc[:pretest_end]

# Use the earliest 60% of the pre-test returns to estimate the band.
calibration_end = int(len(pretest_returns) * 0.60)
calibration_returns = pretest_returns.iloc[:calibration_end]

if calibration_returns.empty:
    raise ValueError("Not enough data to estimate the flat band.")

# Set the band using the 30th percentile of absolute calibration returns.
FLAT_SHARE = 0.30
FLAT_BAND_PCT = calibration_returns.abs().quantile(FLAT_SHARE)

print(
    "Calibration dates:",
    calibration_returns.index.min().date(),
    "to",
    calibration_returns.index.max().date(),
)
print(f"Chosen flat band: -{FLAT_BAND_PCT:.3f}% to +{FLAT_BAND_PCT:.3f}%")

Calibration dates: 2016-10-26 to 2021-08-09
Chosen flat band: -1.321% to +1.321%


Step 4: Create the three-class target

In [12]:
target = pd.Series(np.nan, index=data.index, name="trend_5d")

valid = future_return_pct.notna()

target.loc[
    valid & (future_return_pct < -FLAT_BAND_PCT)
] = -1

target.loc[
    valid & (future_return_pct.abs() <= FLAT_BAND_PCT)
] = 0

target.loc[
    valid & (future_return_pct > FLAT_BAND_PCT)
] = 1

Step 5: Create the seven input features

In [13]:
daily_return = close.pct_change(fill_method=None)

features = pd.DataFrame(index=data.index)

features["return_1d"] = daily_return
features["return_5d"] = close.pct_change(5, fill_method=None)
features["return_20d"] = close.pct_change(20, fill_method=None)

features["volatility_20d"] = daily_return.rolling(20).std()

features["daily_range_pct"] = (data["High"] - data["Low"]) / close

day_range = (data["High"] - data["Low"]).replace(0, np.nan)
features["close_position"] = (close - data["Low"]) / day_range

past_20d_avg_volume = data["Volume"].rolling(20).mean().shift(1)
features["relative_volume"] = data["Volume"] / past_20d_avg_volume

model_data = features.join(target).dropna()

X_all = model_data.drop(columns="trend_5d")
y_all = model_data["trend_5d"].astype(int)

label_names = {-1: "down", 0: "flat", 1: "up"}

print("\nFeature columns:")
print(X_all.columns.tolist())

print("\nClass proportions in available data:")
print(
    y_all.value_counts(normalize=True)
    .reindex([-1, 0, 1])
    .rename(index=label_names)
    .round(3)
)


Feature columns:
['return_1d', 'return_5d', 'return_20d', 'volatility_20d', 'daily_range_pct', 'close_position', 'relative_volume']

Class proportions in available data:
trend_5d
down    0.297
flat    0.323
up      0.379
Name: proportion, dtype: float64


Step 6: Separate development and test data

In [14]:
holdout_position = X_all.index.searchsorted(test_start)

# Leave out five rows before the test period to avoid overlapping target windows.
development_end = max(0, holdout_position - 5)

X_dev = X_all.iloc[:development_end]
y_dev = y_all.iloc[:development_end]

test_mask = X_all.index >= test_start
X_test = X_all.loc[test_mask]
y_test = y_all.loc[test_mask]

if X_test.empty:
    raise ValueError("No test data found for the chosen test start date.")

print(
    "\nDevelopment dates:",
    X_dev.index.min().date(),
    "to",
    X_dev.index.max().date(),
)
print(
    "Final test dates:",
    X_test.index.min().date(),
    "to",
    X_test.index.max().date(),
)

print("\nDevelopment class proportions:")
print(
    y_dev.value_counts(normalize=True)
    .reindex([-1, 0, 1])
    .rename(index=label_names)
    .round(3)
)


Development dates: 2016-11-25 to 2024-10-21
Final test dates: 2024-10-29 to 2026-10-16

Development class proportions:
trend_5d
down    0.290
flat    0.313
up      0.397
Name: proportion, dtype: float64


Step 7: Define the candidate models (Logistic reg and Random Forest)

In [15]:
def make_models():
    return {
        "logistic": Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
            )),
        ]),

        "random_forest": Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("classifier", RandomForestClassifier(
                n_estimators=300,
                max_depth=4,
                min_samples_leaf=30,
                max_features=0.8,
                class_weight="balanced_subsample",
                random_state=42,
                n_jobs=-1,
            )),
        ]),
    }

Step 8: Compare models using chronological validation

In [16]:
cv = TimeSeriesSplit(
    n_splits=5,
    test_size=126,
    gap=5,
)

validation_rows = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X_dev), start=1):
    X_train = X_dev.iloc[train_idx]
    y_train = y_dev.iloc[train_idx]
    X_val = X_dev.iloc[val_idx]
    y_val = y_dev.iloc[val_idx]

    # Baseline predicts the most common class in this fold's training data.
    baseline = DummyClassifier(strategy="most_frequent")
    baseline.fit(X_train, y_train)
    baseline_prediction = baseline.predict(X_val)

    validation_rows.append({
        "fold": fold,
        "model": "baseline",
        "accuracy": accuracy_score(y_val, baseline_prediction),
        "balanced_accuracy": balanced_accuracy_score(
            y_val, baseline_prediction
        ),
        "macro_f1": f1_score(
            y_val,
            baseline_prediction,
            average="macro",
            zero_division=0,
        ),
    })

    for model_name, model in make_models().items():
        model.fit(X_train, y_train)
        prediction = model.predict(X_val)

        validation_rows.append({
            "fold": fold,
            "model": model_name,
            "accuracy": accuracy_score(y_val, prediction),
            "balanced_accuracy": balanced_accuracy_score(
                y_val, prediction
            ),
            "macro_f1": f1_score(
                y_val,
                prediction,
                average="macro",
                zero_division=0,
            ),
        })

validation = pd.DataFrame(validation_rows)

print("\nValidation scores by period:")
print(validation.round(3).to_string(index=False))

summary = (
    validation.groupby("model")[[
        "accuracy",
        "balanced_accuracy",
        "macro_f1",
    ]]
    .mean()
    .sort_values("macro_f1", ascending=False)
)

print("\nAverage validation scores:")
print(summary.round(3).to_string())

candidate_summary = summary.drop(index="baseline")
selected_model_name = candidate_summary["macro_f1"].idxmax()

print("\nSelected model by validation macro F1:", selected_model_name)


Validation scores by period:
 fold         model  accuracy  balanced_accuracy  macro_f1
    1      baseline     0.310              0.333     0.158
    1      logistic     0.357              0.366     0.358
    1 random_forest     0.190              0.205     0.189
    2      baseline     0.278              0.333     0.145
    2      logistic     0.365              0.342     0.292
    2 random_forest     0.389              0.369     0.357
    3      baseline     0.381              0.333     0.184
    3      logistic     0.349              0.362     0.275
    3 random_forest     0.310              0.292     0.275
    4      baseline     0.421              0.333     0.197
    4      logistic     0.341              0.364     0.311
    4 random_forest     0.325              0.289     0.287
    5      baseline     0.310              0.333     0.158
    5      logistic     0.437              0.427     0.423
    5 random_forest     0.389              0.395     0.386

Average validation scores

Step 9: Evaluate both models and print confusion matrices

In [17]:
labels = [-1, 0, 1]
class_names = ["down", "flat", "up"]

# Fit the baseline on all development data.
test_baseline = DummyClassifier(strategy="most_frequent")
test_baseline.fit(X_dev, y_dev)
baseline_test_prediction = test_baseline.predict(X_test)

print("\nBaseline predicts:", label_names[int(test_baseline.classes_[0])])

test_results = []

# Fit each candidate model on development data and evaluate it on test data.
for model_name, model in make_models().items():
    model.fit(X_dev, y_dev)
    prediction = model.predict(X_test)

    test_results.append({
        "model": model_name,
        "accuracy": accuracy_score(y_test, prediction),
        "balanced_accuracy": balanced_accuracy_score(y_test, prediction),
        "macro_f1": f1_score(
            y_test,
            prediction,
            average="macro",
            zero_division=0,
        ),
    })

    print(f"\n{model_name} — confusion matrix")
    print("Rows = actual; columns = predicted. Order: down, flat, up.")
    print(confusion_matrix(y_test, prediction, labels=labels))

    print(f"\n{model_name} — per-class report")
    print(classification_report(
        y_test,
        prediction,
        labels=labels,
        target_names=class_names,
        zero_division=0,
    ))

# Add the baseline test scores for comparison.
test_results.append({
    "model": "baseline",
    "accuracy": accuracy_score(y_test, baseline_test_prediction),
    "balanced_accuracy": balanced_accuracy_score(
        y_test, baseline_test_prediction
    ),
    "macro_f1": f1_score(
        y_test,
        baseline_test_prediction,
        average="macro",
        zero_division=0,
    ),
})

test_summary = pd.DataFrame(test_results).set_index("model")

print("\nFinal test metric comparison:")
print(test_summary.round(3).to_string())

print("\nActual test class proportions:")
print(
    y_test.value_counts(normalize=True)
    .reindex(labels)
    .rename(index=label_names)
    .round(3)
)


Baseline predicts: down

logistic — confusion matrix
Rows = actual; columns = predicted. Order: down, flat, up.
[[ 62  93   5]
 [ 58 104  13]
 [ 46  92  13]]

logistic — per-class report
              precision    recall  f1-score   support

        down       0.37      0.39      0.38       160
        flat       0.36      0.59      0.45       175
          up       0.42      0.09      0.14       151

    accuracy                           0.37       486
   macro avg       0.38      0.36      0.32       486
weighted avg       0.38      0.37      0.33       486


random_forest — confusion matrix
Rows = actual; columns = predicted. Order: down, flat, up.
[[74 58 28]
 [63 77 35]
 [59 73 19]]

random_forest — per-class report
              precision    recall  f1-score   support

        down       0.38      0.46      0.42       160
        flat       0.37      0.44      0.40       175
          up       0.23      0.13      0.16       151

    accuracy                           0.35      

Step 10: Predict a new five-session trend

In [ ]:
# Fit the selected model using all development data.
new_prediction_model = make_models()[selected_model_name]
new_prediction_model.fit(X_dev, y_dev)

# Enter the seven feature values for the prediction date.
# Replace None with values calculated from that date and its past data.
new_feature_values = {
    "return_1d": None,
    "return_5d": None,
    "return_20d": None,
    "volatility_20d": None,
    "daily_range_pct": None,
    "close_position": None,
    "relative_volume": None,
}

# Check that every feature has been entered.
if any(value is None for value in new_feature_values.values()):
    raise ValueError(
        "Enter values for all seven features in new_feature_values."
    )

# Arrange the values in exactly the same column order used during training.
new_features = pd.DataFrame(
    [new_feature_values],
    columns=X_dev.columns,
)

# Predict the class and its model-estimated probabilities.
predicted_class = int(new_prediction_model.predict(new_features)[0])
predicted_probabilities = new_prediction_model.predict_proba(new_features)[0]

class_names = {-1: "down", 0: "flat", 1: "up"}

print("\nNew prediction")
print("Selected model:", selected_model_name)
print("Predicted five-session trend:", class_names[predicted_class])

print("\nModel-estimated class probabilities:")
for class_value, probability in zip(
    new_prediction_model.classes_,
    predicted_probabilities,
):
    print(f"{class_names[int(class_value)]}: {probability:.1%}")